<a href="https://www.kaggle.com/code/alivehues/cse499a-benchmarking?scriptVersionId=315318802" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install -q -U diffusers transformers accelerate safetensors gradio opencv-python-headless pandas openpyxl xformers deepface tf-keras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8

In [2]:

import os, uuid, sqlite3, zipfile, warnings, gc, base64, re
from io import BytesIO
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageDraw, ImageEnhance
import cv2
import torch
import gradio as gr

from diffusers import (
    StableDiffusionImg2ImgPipeline,
    StableDiffusionInpaintPipeline,
    StableDiffusionXLImg2ImgPipeline,
    StableDiffusionXLInpaintPipeline,
    EulerDiscreteScheduler,
)

warnings.filterwarnings("ignore")
from diffusers.utils import logging as dlogging
dlogging.set_verbosity_error()

# ─────────────────────────────────────────────────
# Paths / Settings
# ─────────────────────────────────────────────────
SAVE_DIR        = "/kaggle/working/outputs"
DB_PATH         = "/kaggle/working/history.db"
BENCH_DB_PATH   = "/kaggle/working/benchmark_results.db"
ZIP_PATH        = "/kaggle/working/session_data.zip"
CSV_PATH        = "/kaggle/working/history_export.csv"
XLSX_PATH       = "/kaggle/working/history_export.xlsx"
BENCH_CSV_PATH  = "/kaggle/working/benchmark_export.csv"
BENCH_XLSX_PATH = "/kaggle/working/benchmark_export.xlsx"

os.makedirs(SAVE_DIR, exist_ok=True)

NUM_GPUS = torch.cuda.device_count()
HAS_CUDA = NUM_GPUS > 0
DTYPE    = torch.float16 if HAS_CUDA else torch.float32

_PIPE_CACHE         = {}
_DEVICE_ALLOCATIONS = {}

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

# ─────────────────────────────────────────────────
# DeepFace & VLM Integration
# ─────────────────────────────────────────────────
def deepface_identity_sim(orig_pil: Image.Image, edited_pil: Image.Image) -> dict:
    empty = {
        "identity_sim": float("nan"), "face_found": 0,
        "landmark_shift_mean": float("nan"), "landmark_shift_max": float("nan"),
        "landmark_count_before": 0, "landmark_count_after": 0,
        "identity_changed": -1, "benchmark_note": "DeepFace not available"
    }
    try:
        from deepface import DeepFace
        import tempfile
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as f1:
            orig_path = f1.name
            orig_pil.convert("RGB").save(orig_path, "JPEG")
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as f2:
            edit_path = f2.name
            edited_pil.convert("RGB").save(edit_path, "JPEG")
        result = DeepFace.verify(
            img1_path=orig_path, img2_path=edit_path,
            model_name="ArcFace", detector_backend="opencv",
            enforce_detection=False, silent=True
        )
        os.unlink(orig_path)
        os.unlink(edit_path)
        distance  = float(result["distance"])
        threshold = float(result.get("threshold", 0.68))
        sim       = float(np.clip(1.0 - distance / threshold, 0.0, 1.0))
        verified  = bool(result.get("verified", sim >= 0.5))
        changed   = 0 if verified else 1
        note      = "Identity preserved" if verified else "Identity likely changed"
        return {
            "identity_sim": sim, "face_found": 1,
            "landmark_shift_mean": distance, "landmark_shift_max": threshold,
            "landmark_count_before": 1, "landmark_count_after": 1,
            "identity_changed": changed, "benchmark_note": note
        }
    except Exception as e:
        empty["benchmark_note"] = f"DeepFace error: {str(e)[:80]}"
        return empty

def benchmark_face_change(orig_pil, edited_pil):
    return deepface_identity_sim(orig_pil, edited_pil)

def get_vlm_score(img_pil: Image.Image) -> float:
    """
    Placeholder for actual VLM. 
    Later, load a HuggingFace pipeline here (e.g., LLaVA) to evaluate realism.
    """
    # Simulating a VLM score for DB structure testing
    return round(float(np.random.uniform(2.5, 4.8)), 1)

# ─────────────────────────────────────────────────
# Face detection helpers (masking only)
# ─────────────────────────────────────────────────
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

def detect_face_rect(pil_img):
    gray  = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
    if len(faces) == 0:
        return None
    return max(faces, key=lambda r: r[2] * r[3])

def detect_eyes(pil_img, face_rect):
    if face_rect is None:
        return []
    x, y, w, h = face_rect
    gray = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2GRAY)
    roi  = gray[y:y+h, x:x+w]
    eyes = eye_cascade.detectMultiScale(roi, 1.1, 4)
    return [(x + ex, y + ey, ew, eh) for ex, ey, ew, eh in eyes]

def analyze_image_quality(pil_img: Image.Image) -> dict:
    img_np    = np.array(pil_img.convert("RGB"))
    gray      = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY).astype(np.float32)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_32F).var())
    brightness= float(gray.mean())
    w, h      = pil_img.size
    return {
        "sharpness": sharpness, "brightness": brightness,
        "is_low_res": (w < 512 or h < 512),
        "is_blurry": sharpness < 80.0, "is_dark": brightness < 60.0,
        "width": w, "height": h
    }

def preprocess_for_enhancement(pil_img: Image.Image) -> Image.Image:
    w, h = pil_img.size
    if w < 512 or h < 512:
        scale   = max(512 / w, 512 / h)
        pil_img = pil_img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return pil_img.filter(ImageFilter.UnsharpMask(radius=1.5, percent=80, threshold=3))

# ─────────────────────────────────────────────────
# EDIT INTENT PARSER
# ─────────────────────────────────────────────────
EDIT_RULES = [
    (r"(add|grow|give).*(beard|stubble|mustache|moustache)",
     "lower_face",
     "thick realistic beard, dense facial hair, natural skin texture, visible pores, same gender, same face structure",
     "clean shaven, plastic, artificial, gender change", 0.35),

    (r"(remove|shave|clean).*(beard|stubble|mustache|moustache)",
     "lower_face",
     "clean shaven smooth skin, no facial hair, natural skin texture, visible pores, same gender, same face structure",
     "beard, stubble, mustache, plastic, artificial", 0.30),

    (r"(make|change|give).*(smile|smil|happy|laugh)",
     "lower_face",
     "natural warm smile showing teeth, happy expression, natural skin texture, visible pores, same gender, same face structure",
     "frown, sad, plastic, artificial, morph, identity change", 0.30),

    (r"(make|look).*(sad|depressed|upset|frown)",
     "lower_face",
     "sad expression, frowning, upset, natural skin texture, visible pores, same gender, same face structure",
     "smile, happy, plastic, artificial", 0.30),

    (r"(make|look).*(scared|terrified|fear|shocked)",
     "face",
     "scared expression, wide eyes, fearful, shocked, natural skin texture, visible pores, same gender",
     "calm, neutral, smile, plastic, artificial", 0.35),

    (r"(make|look).*(sleepy|tired|exhausted)",
     "face",
     "sleepy expression, tired eyes, slight under eye bags, natural skin texture, visible pores, same gender",
     "energetic, wide awake, plastic, artificial", 0.30),

    (r"(make|look).*(serious|neutral|stoic)",
     "lower_face",
     "serious neutral expression, closed mouth, natural skin texture, same gender, same face structure",
     "smile, happy, plastic, artificial", 0.28),

    (r"(add|give|change|apply|full|glam).*(makeup|foundation|contour|glam|glamour)",
     "face",
     "full face professional makeup, heavy foundation, contouring, bold lipstick, smoky eye, same gender, same face structure",
     "bare skin, plastic, artificial", 0.50),

    (r"(remove|erase|no|without|clean|bare|natural).*(makeup|lipstick|eyeshadow)",
     "face",
     "completely bare natural skin, no makeup at all, fresh faced, visible pores, same gender",
     "heavy makeup, lipstick, plastic", 0.32),

    (r"(slim|slimmer|thinner|narrow).*(face|jaw|cheek)",
     "face",
     "slim narrow face, sharp defined jawline, high cheekbones, natural skin texture, same gender",
     "chubby, round face, double chin, plastic", 0.45),

    (r"(younger|young|youthful|teen)",
     "face",
     "younger appearance, vibrant skin, natural skin texture, same gender, same identity, same person",
     "heavy makeup, gender swap, female, masculine, plastic, artificial", 0.34),

    (r"(older|old|aged|mature|elderly|wrinkle|age them)",
     "face",
     "older aged appearance, natural wrinkles, age spots, realistic aging skin texture, same gender, same facial structure",
     "young, smooth skin, baby face, plastic, artificial", 0.36),

    (r"(enhance|quality|sharp|detail|4k|8k|hd|high.?res|upscale|realistic|hyper.?real|texture)",
     "ENHANCEMENT",
     "ultra realistic hyperdetailed portrait, 8k RAW photo, sharp natural skin texture, individual hair strands visible, pores, same gender",
     "blurry, low quality, cartoon, watermark, painting, plastic, over-smoothed", 0.30),
]

DEFAULT_EDIT = (
    "full",
    "photorealistic portrait, natural skin texture, same gender, same face",
    "blurry, cartoon, different person, plastic, artificial", 0.22
)

def parse_edit_intent(user_text: str):
    txt = user_text.lower().strip()
    for pattern, region, pos_prompt, neg_extra, strength in EDIT_RULES:
        if re.search(pattern, txt):
            return region, pos_prompt, neg_extra, strength
    return DEFAULT_EDIT

def is_enhancement_request(user_text: str) -> bool:
    txt = user_text.lower().strip()
    for pattern, region, *_ in EDIT_RULES:
        if region == "ENHANCEMENT" and re.search(pattern, txt):
            return True
    return False

# ─────────────────────────────────────────────────
# Mask builders
# ─────────────────────────────────────────────────
def _smooth_mask(mask: Image.Image, blur_radius=18) -> Image.Image:
    return mask.filter(ImageFilter.GaussianBlur(blur_radius))

def build_mask(pil_img: Image.Image, region: str) -> Image.Image:
    W, H  = pil_img.size
    mask  = Image.new("L", (W, H), 0)
    draw  = ImageDraw.Draw(mask)
    face  = detect_face_rect(pil_img)

    if region == "full" or face is None:
        return Image.new("L", (W, H), 255)

    fx, fy, fw, fh = face

    if region == "face":
        pad = int(fw * 0.12)
        draw.rectangle([max(0,fx-pad), max(0,fy-pad), min(W,fx+fw+pad), min(H,fy+fh+pad)], fill=255)

    elif region == "skin":
        cx = fx + fw//2; cy = fy + int(fh*0.55)
        rx = int(fw*0.40); ry = int(fh*0.38)
        draw.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=255)
        draw.rectangle([fx, fy+int(fh*0.20), fx+fw, fy+int(fh*0.45)], fill=0)

    elif region == "eyes":
        eyes = detect_eyes(pil_img, face)
        if eyes:
            for ex, ey, ew, eh in eyes:
                pad = int(max(ew, eh) * 0.55)
                draw.rectangle([max(0,ex-pad), max(0,ey-pad), min(W,ex+ew+pad), min(H,ey+eh+pad)], fill=255)
        else:
            draw.rectangle([fx, fy+int(fh*0.18), fx+fw, fy+int(fh*0.50)], fill=255)

    elif region == "eyes_and_bridge":
        eyes = detect_eyes(pil_img, face)
        if eyes:
            xs  = [ex for ex,_,_,_ in eyes]
            x2s = [ex+ew for ex,_,ew,_ in eyes]
            ys  = [ey for _,ey,_,_ in eyes]
            y2s = [ey+eh for _,ey,_,eh in eyes]
            pad = int(fw * 0.24)
            draw.rectangle([
                max(0, min(xs)-pad),
                max(0, min(ys)-int(fh*0.12)),
                min(W, max(x2s)+pad),
                min(H, max(y2s)+int(fh*0.20))
            ], fill=255)
        else:
            draw.rectangle([fx, fy+int(fh*0.08), fx+fw, fy+int(fh*0.52)], fill=255)

    elif region == "lower_face":
        mouth_top = fy + int(fh * 0.52)
        bottom    = min(H, fy + fh + int(fh * 0.25))
        draw.rectangle([max(0, fx-int(fw*0.05)), mouth_top,
                        min(W, fx+fw+int(fw*0.05)), bottom], fill=255)

    if np.array(mask).max() == 0:
        return Image.new("L", (W, H), 255)

    return _smooth_mask(mask, blur_radius=14)

# ─────────────────────────────────────────────────
# DB init
# ─────────────────────────────────────────────────
def init_history_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS source_faces (
            source_id TEXT PRIMARY KEY, dataset_name TEXT,
            image_path TEXT UNIQUE, file_name TEXT,
            added_timestamp TEXT, is_active INTEGER DEFAULT 1
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS results (
            id TEXT PRIMARY KEY, session_id TEXT,
            source_id TEXT, input_mode TEXT, iteration INTEGER,
            base_prompt TEXT, refined_prompt TEXT, realism_score REAL,
            vlm_score REAL, scorer_id TEXT,
            model_name TEXT, image_path TEXT, timestamp TEXT,
            identity_sim REAL, face_found INTEGER,
            landmark_shift_mean REAL, landmark_shift_max REAL,
            landmark_count_before INTEGER, landmark_count_after INTEGER,
            identity_changed INTEGER, benchmark_note TEXT
        )
    """)
    conn.commit()
    conn.close()

def init_benchmark_db():
    conn = sqlite3.connect(BENCH_DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS benchmark_sources (
            source_id TEXT PRIMARY KEY, dataset_name TEXT,
            image_path TEXT, file_name TEXT,
            first_seen_timestamp TEXT, last_updated_timestamp TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS benchmark_current (
            source_id TEXT NOT NULL, model_name TEXT NOT NULL,
            dataset_name TEXT, file_name TEXT, image_path TEXT,
            latest_result_id TEXT, latest_prompt TEXT, latest_score REAL,
            latest_vlm_score REAL, latest_scorer_id TEXT,
            latest_iteration INTEGER, latest_output_path TEXT, latest_timestamp TEXT,
            latest_identity_sim REAL, latest_face_found INTEGER,
            latest_landmark_shift_mean REAL, latest_landmark_shift_max REAL,
            latest_landmark_count_before INTEGER, latest_landmark_count_after INTEGER,
            latest_identity_changed INTEGER, latest_note TEXT,
            run_count INTEGER DEFAULT 0,
            best_score REAL, best_iteration INTEGER, best_output_path TEXT, best_timestamp TEXT,
            PRIMARY KEY (source_id, model_name)
        )
    """)
    conn.commit()
    conn.close()

init_history_db()
init_benchmark_db()

# ─────────────────────────────────────────────────
# Dataset / DB helpers
# ─────────────────────────────────────────────────
def find_first_image_folder(root="/kaggle/input/datasets/sathyavgc/celeba-dataset-small/content/dataset_sample/images"):
    # Fallback if specific dataset folder doesn't exist
    if not os.path.exists(root):
        root = "/kaggle/input" 
    best_folder, best_count = None, 0
    for current_root, _, files in os.walk(root):
        count = sum(1 for f in files if os.path.splitext(f.lower())[1] in IMG_EXTS)
        if count > best_count:
            best_count = count; best_folder = current_root
    return best_folder, best_count

def count_source_faces():
    conn  = sqlite3.connect(DB_PATH)
    total = conn.execute("SELECT COUNT(*) FROM source_faces WHERE is_active=1").fetchone()[0]
    conn.close()
    return total

def prepare_source_face_db(limit=5000):
    existing = count_source_faces()
    if existing > 0:
        return f"✅ DB exists. **{existing}** rows. Click **LOAD RANDOM FACE**."
    folder, img_count = find_first_image_folder()
    if folder is None or img_count == 0:
        return "❌ No image folder found inside /kaggle/input."
    files = [
        os.path.join(folder, name) for name in sorted(os.listdir(folder))
        if os.path.isfile(os.path.join(folder, name))
        and os.path.splitext(name.lower())[1] in IMG_EXTS
    ][:limit]
    if not files: return f"❌ No usable image files found in: {folder}"
    dataset_name = os.path.basename(folder)
    conn = sqlite3.connect(DB_PATH)
    for path in files:
        try:
            source_id = str(uuid.uuid5(uuid.NAMESPACE_URL, path))
            conn.execute("""
                INSERT OR IGNORE INTO source_faces
                (source_id, dataset_name, image_path, file_name, added_timestamp, is_active)
                VALUES (?,?,?,?,?,1)
            """, (source_id, dataset_name, path, os.path.basename(path), datetime.now().isoformat()))
        except: pass
    conn.commit()
    total = conn.execute("SELECT COUNT(*) FROM source_faces WHERE is_active=1").fetchone()[0]
    conn.close()
    return f"✅ DB created. **{total}** rows. Now click **LOAD RANDOM FACE**."

def load_random_source_face():
    conn = sqlite3.connect(DB_PATH)
    row  = conn.execute("""
        SELECT source_id, image_path, dataset_name, file_name
        FROM source_faces WHERE is_active=1 ORDER BY RANDOM() LIMIT 1
    """).fetchone()
    conn.close()
    if row is None:
        return None, None, "⚠️ No source faces. Click PREPARE FACE DB first."
    source_id, image_path, dataset_name, file_name = row
    try:
        img  = Image.open(image_path).convert("RGB")
        meta = {"source_id": source_id, "dataset_name": dataset_name,
                "file_name": file_name, "image_path": image_path}
        return img, meta, f"✅ Random face loaded.\n\n**File:** {file_name}\n\n**Source ID:** `{source_id}`"
    except Exception as e:
        return None, None, f"❌ Could not open source image: {e}"

# ─────────────────────────────────────────────────
# Benchmark DB helpers
# ─────────────────────────────────────────────────
def register_benchmark_source(source_id, dataset_name, image_path, file_name):
    if not source_id: return
    now_ts = datetime.now().isoformat()
    conn   = sqlite3.connect(BENCH_DB_PATH)
    exists = conn.execute("SELECT source_id FROM benchmark_sources WHERE source_id=?", (source_id,)).fetchone()
    if exists is None:
        conn.execute("""
            INSERT INTO benchmark_sources
            (source_id, dataset_name, image_path, file_name, first_seen_timestamp, last_updated_timestamp)
            VALUES (?,?,?,?,?,?)
        """, (source_id, dataset_name, image_path, file_name, now_ts, now_ts))
    else:
        conn.execute(
            "UPDATE benchmark_sources SET dataset_name=?,image_path=?,file_name=?,last_updated_timestamp=? WHERE source_id=?",
            (dataset_name, image_path, file_name, now_ts, source_id)
        )
    conn.commit(); conn.close()

def upsert_benchmark_current(source_meta, model_name, result_payload, score_value, scorer_id):
    if not source_meta or not source_meta.get("source_id"):
        return "ℹ️ Benchmark DB update skipped (not from dataset DB)."
    source_id    = source_meta["source_id"]
    dataset_name = source_meta.get("dataset_name")
    image_path   = source_meta.get("image_path")
    file_name    = source_meta.get("file_name")
    register_benchmark_source(source_id, dataset_name, image_path, file_name)
    conn = sqlite3.connect(BENCH_DB_PATH)
    existing = conn.execute("""
        SELECT best_score, best_iteration, best_output_path, best_timestamp, run_count
        FROM benchmark_current WHERE source_id=? AND model_name=?
    """, (source_id, model_name)).fetchone()
    
    if existing is None:
        best_score = float(score_value); best_iter = int(result_payload["iteration"])
        best_path  = result_payload["image_path"]; best_ts = result_payload["timestamp"]; run_count = 1
    else:
        prev_best, prev_best_iter, prev_best_path, prev_best_ts, prev_rc = existing
        run_count = int(prev_rc or 0) + 1
        if prev_best is None or float(score_value) >= float(prev_best):
            best_score = float(score_value); best_iter = int(result_payload["iteration"])
            best_path  = result_payload["image_path"]; best_ts = result_payload["timestamp"]
        else:
            best_score, best_iter, best_path, best_ts = prev_best, prev_best_iter, prev_best_path, prev_best_ts
            
    conn.execute("""
        INSERT INTO benchmark_current (
            source_id, model_name, dataset_name, file_name, image_path,
            latest_result_id, latest_prompt, latest_score, latest_vlm_score, latest_scorer_id, latest_iteration,
            latest_output_path, latest_timestamp, latest_identity_sim, latest_face_found,
            latest_landmark_shift_mean, latest_landmark_shift_max,
            latest_landmark_count_before, latest_landmark_count_after,
            latest_identity_changed, latest_note,
            run_count, best_score, best_iteration, best_output_path, best_timestamp
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        ON CONFLICT(source_id, model_name) DO UPDATE SET
            dataset_name=excluded.dataset_name, file_name=excluded.file_name,
            image_path=excluded.image_path,
            latest_result_id=excluded.latest_result_id,
            latest_prompt=excluded.latest_prompt, latest_score=excluded.latest_score,
            latest_vlm_score=excluded.latest_vlm_score, latest_scorer_id=excluded.latest_scorer_id,
            latest_iteration=excluded.latest_iteration,
            latest_output_path=excluded.latest_output_path,
            latest_timestamp=excluded.latest_timestamp,
            latest_identity_sim=excluded.latest_identity_sim,
            latest_face_found=excluded.latest_face_found,
            latest_landmark_shift_mean=excluded.latest_landmark_shift_mean,
            latest_landmark_shift_max=excluded.latest_landmark_shift_max,
            latest_landmark_count_before=excluded.latest_landmark_count_before,
            latest_landmark_count_after=excluded.latest_landmark_count_after,
            latest_identity_changed=excluded.latest_identity_changed,
            latest_note=excluded.latest_note,
            run_count=excluded.run_count, best_score=excluded.best_score,
            best_iteration=excluded.best_iteration,
            best_output_path=excluded.best_output_path, best_timestamp=excluded.best_timestamp
    """, (
        source_id, model_name, dataset_name, file_name, image_path,
        result_payload["id"], result_payload["refined_prompt"], float(score_value),
        float(result_payload.get("vlm_score", 0.0)), scorer_id,
        int(result_payload["iteration"]), result_payload["image_path"], result_payload["timestamp"],
        result_payload["identity_sim"], result_payload["face_found"],
        result_payload["landmark_shift_mean"], result_payload["landmark_shift_max"],
        result_payload["landmark_count_before"], result_payload["landmark_count_after"],
        result_payload["identity_changed"], result_payload["benchmark_note"],
        run_count, best_score, best_iter, best_path, best_ts
    ))
    conn.commit(); conn.close()
    return f"✅ Benchmark DB updated for model `{model_name}`."

# ─────────────────────────────────────────────────
# Model registry
# ─────────────────────────────────────────────────
MODEL_REGISTRY = {
    "Juggernaut-XL-v9": {
        "repo_id": "RunDiffusion/Juggernaut-XL-v9",
        "inpaint_repo": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        "size": (1024, 1024), "steps_ui": 30, "guidance": 7.0, "is_xl": True
    },
    "SD-1.5": {
        "repo_id": "runwayml/stable-diffusion-v1-5",
        "inpaint_repo": "runwayml/stable-diffusion-inpainting",
        "size": (512, 512), "steps_ui": 28, "guidance": 7.5, "is_xl": False
    },
    "Realistic-Vision-V5.1": {
        "repo_id": "SG161222/Realistic_Vision_V5.1_noVAE",
        "inpaint_repo": "Uminosachi/realisticVisionV51_v51VAE-inpainting",
        "size": (512, 512), "steps_ui": 28, "guidance": 7.0, "is_xl": False
    },
    "epiCRealism": {
        "repo_id": "emilianJR/epiCRealism",
        "inpaint_repo": "runwayml/stable-diffusion-inpainting",
        "size": (512, 512), "steps_ui": 28, "guidance": 7.0, "is_xl": False
    },
}

# ─────────────────────────────────────────────────
# Prompt categories
# ─────────────────────────────────────────────────
PROMPT_CATEGORIES = {
    "expressions": [
        "make this person smile",
        "make this person look sad",
        "make this person look scared",
        "make this person look sleepy",
        "serious neutral expression",
    ],
    "facial hair": [
        "add beard",
        "remove beard",
    ],
    "makeup": [
        "add full glamour makeup",
        "remove makeup completely",
    ],
    "face shape": [
        "slim the face",
    ],
    "age modification": [
        "make them look younger",
        "make them look older",
    ],
    "quality enhancement": [
        "enhance to hyperrealistic 4k resolution",
    ],
}

# ─────────────────────────────────────────────────
# Pipeline cache
# ─────────────────────────────────────────────────
def _get_device():
    if not HAS_CUDA: return "cpu"
    gpu_counts = {i: list(_DEVICE_ALLOCATIONS.values()).count(f"cuda:{i}") for i in range(NUM_GPUS)}
    return f"cuda:{min(gpu_counts, key=gpu_counts.get)}"

def get_pipe(model_name: str, mode="img2img"):
    cache_key = f"{model_name}|{mode}"
    if cache_key in _PIPE_CACHE:
        return _PIPE_CACHE[cache_key], _DEVICE_ALLOCATIONS[cache_key]
    max_cache = max(1, NUM_GPUS)
    if len(_PIPE_CACHE) >= max_cache:
        _PIPE_CACHE.clear(); _DEVICE_ALLOCATIONS.clear()
        gc.collect()
        if HAS_CUDA: torch.cuda.empty_cache()
    device = _get_device()
    cfg    = MODEL_REGISTRY[model_name]
    is_xl  = cfg["is_xl"]
    if mode == "inpaint":
        repo = cfg["inpaint_repo"]
        pipe = (StableDiffusionXLInpaintPipeline if is_xl else StableDiffusionInpaintPipeline).from_pretrained(
            repo, torch_dtype=DTYPE,
            **(dict(variant="fp16") if HAS_CUDA and is_xl else {}),
            use_safetensors=True
        )
    else:
        repo = cfg["repo_id"]
        pipe = (StableDiffusionXLImg2ImgPipeline if is_xl else StableDiffusionImg2ImgPipeline).from_pretrained(
            repo, torch_dtype=DTYPE,
            **(dict(variant="fp16") if HAS_CUDA and is_xl else {}),
            use_safetensors=True
        )
    pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.to(device)
    if HAS_CUDA:
        try: pipe.enable_xformers_memory_efficient_attention()
        except: pipe.enable_attention_slicing()
    _PIPE_CACHE[cache_key] = pipe; _DEVICE_ALLOCATIONS[cache_key] = device
    return pipe, device

# ─────────────────────────────────────────────────
# Generation Logic
# ─────────────────────────────────────────────────
def generate_enhancement(orig_image, user_prompt, model_name, seed=42):
    cfg = MODEL_REGISTRY[model_name]
    W, H = cfg["size"]
    quality     = analyze_image_quality(orig_image)
    preprocessed= preprocess_for_enhancement(orig_image)
    resized     = preprocessed.resize((W, H), Image.LANCZOS)
    _, pos_prompt, neg_extra, _ = parse_edit_intent(user_prompt)
    identity_anchor = (
        "same person, exact same gender, preserve identity, preserve facial features, "
        "preserve skin tone, natural skin texture, visible pores"
    )
    final_pos = f"{pos_prompt}, {identity_anchor}"
    final_neg = (
        "blurry, low quality, cartoon, watermark, different person, gender change, "
        "face swap, identity change, anime, illustration, plastic, smooth skin"
        + (f", {neg_extra}" if neg_extra else "")
    )
    strength  = 0.38 if (quality["is_blurry"] or quality["is_low_res"]) else 0.28
    device    = _get_device()
    dev_str   = "cuda" if "cuda" in device else "cpu"
    g         = torch.Generator(device=device).manual_seed(int(seed))
    pipe, device = get_pipe(model_name, mode="img2img")
    ctx = torch.autocast(dev_str, dtype=DTYPE) if dev_str == "cuda" else torch.no_grad()
    with ctx:
        out = pipe(
            prompt=final_pos, negative_prompt=final_neg,
            image=resized, strength=strength,
            num_inference_steps=int(cfg["steps_ui"]) + 5,
            guidance_scale=float(cfg["guidance"]), generator=g
        ).images[0]
    orig_np = np.array(resized, dtype=np.float32)
    out_np  = np.array(out.convert("RGB"), dtype=np.float32)
    blended = (0.70 * out_np + 0.30 * orig_np).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended), final_pos, final_neg, strength

def generate_identity_preserving(orig_image, user_prompt, model_name, seed=42):
    cfg   = MODEL_REGISTRY[model_name]
    W, H  = cfg["size"]
    mask_region, inpaint_pos, neg_extra, strength = parse_edit_intent(user_prompt)
    resized_orig = orig_image.resize((W, H), Image.LANCZOS)
    mask         = build_mask(resized_orig, mask_region)
    identity_anchor = (
        "same person, exact same gender, preserve identity, preserve facial features, "
        "natural skin texture, visible pores, realistic portrait photo, photorealistic"
    )
    final_pos = f"{inpaint_pos}, {identity_anchor}"
    final_neg = (
        "blurry, low quality, cartoon, watermark, text, deformed, plastic, smooth skin, "
        "different person, new face, face swap, identity change, gender change, "
        "anime, illustration"
        + (f", {neg_extra}" if neg_extra else "")
    )
    device    = _get_device()
    dev_str   = "cuda" if "cuda" in device else "cpu"
    g         = torch.Generator(device=device).manual_seed(int(seed))
    try:
        pipe, device = get_pipe(model_name, mode="inpaint")
        ctx = torch.autocast(dev_str, dtype=DTYPE) if dev_str == "cuda" else torch.no_grad()
        with ctx:
            out = pipe(
                prompt=final_pos, negative_prompt=final_neg,
                image=resized_orig, mask_image=mask,
                strength=float(strength),
                num_inference_steps=int(cfg["steps_ui"]),
                guidance_scale=float(cfg["guidance"]), generator=g
            ).images[0]
    except Exception as e:
        pipe, device = get_pipe(model_name, mode="img2img")
        ctx = torch.autocast(dev_str, dtype=DTYPE) if dev_str == "cuda" else torch.no_grad()
        with ctx:
            out = pipe(
                prompt=final_pos, negative_prompt=final_neg,
                image=resized_orig, strength=min(strength, 0.22),
                num_inference_steps=int(cfg["steps_ui"]),
                guidance_scale=float(cfg["guidance"]), generator=g
            ).images[0]
        return out, final_pos, final_neg, mask_region, strength
    alpha      = np.array(mask, dtype=np.float32)[:,:,None] / 255.0
    orig_np    = np.array(resized_orig, dtype=np.float32)
    out_np     = np.array(out.convert("RGB"), dtype=np.float32)
    blended_np = (alpha * out_np + (1.0 - alpha) * orig_np).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended_np), final_pos, final_neg, mask_region, strength

# ─────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────
def ensure_pil(img):
    if img is None: return None
    if isinstance(img, dict):
        img = img.get("background") or img.get("layers")[0]
    if isinstance(img, np.ndarray):
        return Image.fromarray(img.astype("uint8")).convert("RGB")
    if isinstance(img, Image.Image):
        return img.convert("RGB")
    return None

def thumb_b64(img_path, size=(260, 260)):
    try:
        img = Image.open(img_path).convert("RGB"); img.thumbnail(size)
        bio = BytesIO(); img.save(bio, format="JPEG", quality=90)
        return base64.b64encode(bio.getvalue()).decode("utf-8")
    except: return ""

def full_b64(img_path, size=(1800, 1800)):
    try:
        img = Image.open(img_path).convert("RGB"); img.thumbnail(size)
        bio = BytesIO(); img.save(bio, format="JPEG", quality=95)
        return base64.b64encode(bio.getvalue()).decode("utf-8")
    except: return ""

def esc(text):
    if text is None: return ""
    return (str(text).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
            .replace('"',"&quot;").replace("'","&#39;").replace("\n","&#10;"))

# ─────────────────────────────────────────────────
# History modal HTML
# ─────────────────────────────────────────────────
def load_history_modal_html(limit=20):
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("""
        SELECT image_path, model_name, realism_score, timestamp, id,
               refined_prompt, identity_sim, face_found, iteration,
               input_mode, landmark_shift_mean, landmark_shift_max,
               landmark_count_before, landmark_count_after, identity_changed, benchmark_note, vlm_score, scorer_id
        FROM results ORDER BY timestamp DESC LIMIT ?
    """, (limit,)).fetchall()
    conn.close()

    if not rows:
        return "<div style='color:#aaa;padding:12px;'>No history yet.</div>"

    best_idx, best_score = -1, -1
    for i, r in enumerate(rows):
        s = -1 if r[2] is None else float(r[2])
        if s > best_score: best_score = s; best_idx = i

    cards = ['<div class="history-grid">']
    for i, r in enumerate(rows):
        (image_path, model_name, realism_score, timestamp, run_id,
         refined_prompt, identity_sim, face_found, iteration,
         input_mode, shift_mean, shift_max, count_before, count_after,
         identity_changed, benchmark_note, vlm_score, scorer_id) = r
        tb        = thumb_b64(image_path)
        fb        = full_b64(image_path)
        score_txt = "Pending" if realism_score is None else f"{float(realism_score):.1f}/5"
        id_txt    = f"{float(identity_sim):.3f}" if face_found==1 and str(identity_sim)!="nan" else "N/A"
        sm_txt    = f"{float(shift_mean):.4f}" if face_found==1 and str(shift_mean)!="nan" else "N/A"
        sx_txt    = f"{float(shift_max):.4f}" if face_found==1 and str(shift_max)!="nan" else "N/A"
        changed   = "Yes" if identity_changed==1 else "No" if identity_changed==0 else "Unknown"
        badge     = '<div class="best-badge">⭐ BEST</div>' if i==best_idx and best_score>=0 else ""
        cards.append(f"""
        <div class="history-card" onclick="openHistoryModal(this)"
             data-img="data:image/jpeg;base64,{fb}"
             data-model="{esc(model_name)}" data-score="{esc(score_txt)}"
             data-time="{esc(timestamp)}" data-runid="{esc(run_id)}"
             data-prompt="{esc(refined_prompt)}" data-identity="{esc(id_txt)}"
             data-shiftmean="{esc(sm_txt)}" data-shiftmax="{esc(sx_txt)}"
             data-mode="{esc(input_mode)}" data-iteration="{esc(iteration)}"
             data-landmarksbefore="{esc(count_before)}" data-landmarksafter="{esc(count_after)}"
             data-changed="{esc(changed)}" data-note="{esc(benchmark_note)}"
             data-vlm="{esc(vlm_score)}" data-scorer="{esc(scorer_id)}">
             {badge}
             <img src="data:image/jpeg;base64,{tb}" alt="history">
             <div class="card-info">
                 <div class="card-model">{esc(model_name)}</div>
                 <div class="card-score">Score: {esc(score_txt)}</div>
             </div>
        </div>""")
    cards.append("</div>")
    cards.append("""
    <div id="historyModal" class="history-modal" onclick="closeHistoryModal(event)">
      <div class="history-modal-content" onclick="event.stopPropagation()">
        <button class="history-close-btn" onclick="forceCloseHistoryModal()">&times;</button>
        <div class="history-modal-layout">
          <div class="history-modal-left"><img id="hmImg" class="history-modal-image" src="" alt="preview"></div>
          <div class="history-modal-right">
            <h3>RUN DETAILS</h3>
            <div><b>Scorer:</b> <span id="hmScorer"></span></div>
            <div><b>Model:</b> <span id="hmModel"></span></div>
            <div><b>Realism Score:</b> <span id="hmScore"></span></div>
            <div><b>VLM Score:</b> <span id="hmVlm"></span></div>
            <div><b>Timestamp:</b> <span id="hmTime"></span></div>
            <div><b>Run ID:</b> <span id="hmRunId"></span></div>
            <div><b>Input Mode:</b> <span id="hmMode"></span></div>
            <div><b>Iteration:</b> <span id="hmIteration"></span></div>
            <div><b>Identity Similarity (ArcFace):</b> <span id="hmIdentity"></span></div>
            <div><b>ArcFace Distance:</b> <span id="hmShiftMean"></span></div>
            <div><b>ArcFace Threshold:</b> <span id="hmShiftMax"></span></div>
            <div><b>Identity Changed?:</b> <span id="hmChanged"></span></div>
            <div><b>Benchmark Note:</b> <span id="hmNote"></span></div>
            <div style="margin-top:10px;"><b>Prompt Used:</b></div>
            <div id="hmPrompt" class="history-prompt-box"></div>
          </div>
        </div>
      </div>
    </div>
    <script>
    function openHistoryModal(card) {
        document.getElementById("hmImg").src = card.dataset.img||"";
        document.getElementById("hmModel").textContent = card.dataset.model||"";
        document.getElementById("hmScore").textContent = card.dataset.score||"";
        document.getElementById("hmTime").textContent = card.dataset.time||"";
        document.getElementById("hmRunId").textContent = card.dataset.runid||"";
        document.getElementById("hmMode").textContent = card.dataset.mode||"";
        document.getElementById("hmIteration").textContent = card.dataset.iteration||"";
        document.getElementById("hmIdentity").textContent = card.dataset.identity||"";
        document.getElementById("hmShiftMean").textContent = card.dataset.shiftmean||"";
        document.getElementById("hmShiftMax").textContent = card.dataset.shiftmax||"";
        document.getElementById("hmChanged").textContent = card.dataset.changed||"";
        document.getElementById("hmNote").textContent = card.dataset.note||"";
        document.getElementById("hmPrompt").textContent = card.dataset.prompt||"";
        document.getElementById("hmVlm").textContent = card.dataset.vlm||"Pending";
        document.getElementById("hmScorer").textContent = card.dataset.scorer||"Anonymous";
        document.getElementById("historyModal").style.display="flex";
        document.body.style.overflow="hidden";
    }
    function forceCloseHistoryModal() {
        const m=document.getElementById("historyModal");
        if(m) m.style.display="none";
        document.body.style.overflow="auto";
    }
    function closeHistoryModal(event) {
        if(event.target&&event.target.id==="historyModal") forceCloseHistoryModal();
    }
    document.addEventListener("keydown",e=>{if(e.key==="Escape") forceCloseHistoryModal();});
    </script>""")
    return "".join(cards)

# ─────────────────────────────────────────────────
# Export
# ─────────────────────────────────────────────────
def export_data():
    try:
        conn = sqlite3.connect(DB_PATH)
        results_df = pd.read_sql_query("SELECT * FROM results ORDER BY timestamp DESC", conn)
        source_df  = pd.read_sql_query("SELECT * FROM source_faces ORDER BY added_timestamp DESC", conn)
        conn.close()
        bc = sqlite3.connect(BENCH_DB_PATH)
        bsrc_df = pd.read_sql_query("SELECT * FROM benchmark_sources ORDER BY last_updated_timestamp DESC", bc)
        bcur_df = pd.read_sql_query("SELECT * FROM benchmark_current ORDER BY latest_timestamp DESC", bc)
        bc.close()
        results_df.to_csv(CSV_PATH, index=False)
        bcur_df.to_csv(BENCH_CSV_PATH, index=False)
        with pd.ExcelWriter(XLSX_PATH, engine="openpyxl") as w:
            results_df.to_excel(w, sheet_name="results", index=False)
            source_df.to_excel(w, sheet_name="source_faces", index=False)
        with pd.ExcelWriter(BENCH_XLSX_PATH, engine="openpyxl") as w:
            bsrc_df.to_excel(w, sheet_name="benchmark_sources", index=False)
            bcur_df.to_excel(w, sheet_name="benchmark_current", index=False)
        if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
            for p, n in [(DB_PATH,"history.db"),(BENCH_DB_PATH,"benchmark_results.db"),
                         (CSV_PATH,"history_export.csv"),(XLSX_PATH,"history_export.xlsx"),
                         (BENCH_CSV_PATH,"benchmark_export.csv"),(BENCH_XLSX_PATH,"benchmark_export.xlsx")]:
                if os.path.exists(p): zf.write(p, n)
            for root, _, files in os.walk(SAVE_DIR):
                for fn in files:
                    fp = os.path.join(root, fn)
                    zf.write(fp, os.path.join("outputs", fn))
        return gr.update(value=ZIP_PATH, visible=True)
    except Exception as e:
        print(f"Export error: {e}")
        return gr.update(value=None, visible=False)

# ─────────────────────────────────────────────────
# Auto-refinement prompt boosters
# ─────────────────────────────────────────────────
PROMPT_BOOSTERS = [
    ", highly detailed, sharp focus, natural skin texture, highly realistic",
    ", ultra realistic 8k, perfect skin texture, professional photography, visible pores",
    ", hyperrealistic masterpiece, absolute flawless detail, highly realistic skin",
    ", maximum realism, studio quality, crystal clear, natural lighting, 8k RAW",
    ", absolute photorealism, magazine cover quality, ultra sharp, natural subsurface skin scattering",
]

# ─────────────────────────────────────────────────
# Main iterate / save / refine actions (Editor Mode)
# ─────────────────────────────────────────────────
def iterate_action(input_mode, current_image, user_prompt, model_name, realism_score, feedback, scorer_id, state):
    current_image = ensure_pil(current_image)
    state = dict(state or {})

    if current_image is None:
        return (None, state, "⚠️ Upload an image or load one from DB first.",
                gr.update(visible=False), gr.update(visible=False),
                "No runs executed yet.", "Face Identity Tracker: **—**", load_history_modal_html())

    if not state:
        state = {
            "session_id": str(uuid.uuid4()), "iteration": 1,
            "orig_image": current_image.copy(), "source_id": None,
            "input_mode": input_mode, "dataset_name": None,
            "file_name": None, "image_path": None,
            "last_result_id": None, "last_result_payload": None, "score_saved": False
        }
    else:
        for k, v in [("session_id",str(uuid.uuid4())),("iteration",1),
                     ("orig_image",current_image.copy()),("source_id",None),
                     ("input_mode",input_mode),("dataset_name",None),("file_name",None),
                     ("image_path",None),("last_result_id",None),("last_result_payload",None),("score_saved",False)]:
            state.setdefault(k, v)

    if state.get("orig_image") is None:
        state["orig_image"] = current_image.copy()

    source_id    = state.get("source_id")
    dataset_name = state.get("dataset_name")
    file_name    = state.get("file_name")
    image_path   = state.get("image_path")
    state["input_mode"] = input_mode

    orig_image = state["orig_image"]
    session_id = state["session_id"]
    iteration  = int(state.get("iteration", 1))

    combined_prompt = user_prompt
    if feedback and feedback.strip():
        combined_prompt = f"{user_prompt}, {feedback.strip()}"

    mode_label = "🎯 Identity-Preserving Edit"
    if is_enhancement_request(combined_prompt):
        out, refined_prompt, neg_prompt, strength = generate_enhancement(
            orig_image if iteration == 1 else current_image,
            combined_prompt, model_name, seed=42 + iteration
        )
        mask_region = "enhancement"
        mode_label  = "✨ Quality Enhancement"
    else:
        out, refined_prompt, neg_prompt, mask_region, strength = generate_identity_preserving(
            orig_image if iteration == 1 else current_image,
            combined_prompt, model_name, seed=42 + iteration
        )

    bench = benchmark_face_change(orig_image, out)
    vlm_score = get_vlm_score(out)
    uid   = str(uuid.uuid4())[:8]
    path  = os.path.join(SAVE_DIR, f"{uid}.png")
    out.save(path)
    now_ts = datetime.now().isoformat()

    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        INSERT INTO results (
            id, session_id, source_id, input_mode, iteration,
            base_prompt, refined_prompt, realism_score, vlm_score, scorer_id,
            model_name, image_path, timestamp,
            identity_sim, face_found, landmark_shift_mean, landmark_shift_max,
            landmark_count_before, landmark_count_after, identity_changed, benchmark_note
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        uid, session_id, source_id, input_mode, iteration,
        combined_prompt, refined_prompt, None, vlm_score, scorer_id,
        model_name, path, now_ts,
        bench["identity_sim"], bench["face_found"],
        bench["landmark_shift_mean"], bench["landmark_shift_max"],
        bench["landmark_count_before"], bench["landmark_count_after"],
        bench["identity_changed"], bench["benchmark_note"]
    ))
    conn.commit(); conn.close()

    result_payload = {
        "id": uid, "session_id": session_id, "source_id": source_id,
        "input_mode": input_mode, "iteration": iteration,
        "base_prompt": combined_prompt, "refined_prompt": refined_prompt,
        "model_name": model_name, "image_path": path, "timestamp": now_ts, "vlm_score": vlm_score, **bench
    }
    state.update({
        "iteration": iteration + 1, "last_result_id": uid,
        "last_result_payload": result_payload, "score_saved": False,
        "source_id": source_id, "dataset_name": dataset_name,
        "file_name": file_name, "image_path": image_path,
        "input_mode": input_mode, "orig_image": orig_image
    })

    if bench["face_found"] == 1:
        sim_v = bench["identity_sim"]
        face_status = f"Face Identity Tracker: **✅** — ArcFace similarity: `{sim_v:.3f}`"
        face_status += " ✔️ Same person" if bench["identity_changed"] == 0 else " ⚠️ Identity may have drifted"
    else:
        face_status = "Face Identity Tracker: **❌ Face not detected**"

    sim_display  = f"`{bench['identity_sim']:.3f}`" if bench['face_found'] == 1 else "`N/A`"
    dist_display = f"`{bench['landmark_shift_mean']:.4f}` (ArcFace distance)" if bench['face_found'] == 1 else "`N/A`"

    details = f"""
**Mode:** {mode_label}

**Model Engine:** {model_name}

**Edit / Mask Region:** `{mask_region}`

**Inpainting Strength:** `{strength}`

**Executed Prompt:** {refined_prompt}

**Negative Prompt:** {neg_prompt[:120]}...

**ArcFace Identity Similarity:** {sim_display}

**Identity Changed?:** `{'Yes' if bench['identity_changed']==1 else 'No' if bench['identity_changed']==0 else 'Unknown'}`

**VLM Generated Score:** `{vlm_score}/5`
"""
    return (
        out, state,
        f"✅ Iteration {iteration} complete. Give it a score (0-5) and submit.",
        gr.update(visible=True),
        gr.update(visible=True, value=3),
        details, face_status, load_history_modal_html()
    )

def submit_and_refine_action(input_mode, current_image, user_prompt, model_name, score_value, feedback, scorer_id, state):
    state      = dict(state or {})
    cur_image  = ensure_pil(current_image)
    int_score  = int(round(float(score_value)))
    int_score  = max(0, min(5, int_score))

    if state.get("last_result_id") and state.get("last_result_payload"):
        result_id = state["last_result_id"]
        payload   = state["last_result_payload"]
        conn = sqlite3.connect(DB_PATH)
        conn.execute("UPDATE results SET realism_score=?, scorer_id=? WHERE id=?", (float(int_score), scorer_id, result_id))
        conn.commit(); conn.close()
        source_meta = {
            "source_id":    state.get("source_id"),
            "dataset_name": state.get("dataset_name"),
            "file_name":    state.get("file_name"),
            "image_path":   state.get("image_path"),
        }
        upsert_benchmark_current(source_meta, payload["model_name"], payload, float(int_score), scorer_id)
        state["score_saved"] = True
        payload["realism_score"] = float(int_score)
        state["last_result_payload"] = payload

    if int_score >= 4:
        face_st = state.get("last_face_status", "Face Identity Tracker: **—**")
        details = (
            f"**✅ Score {int_score}/5 saved — output accepted!**\n\n"
            f"**Model Engine:** {state.get('last_result_payload', {}).get('model_name', model_name)}\n\n"
            f"**Final Prompt:** {state.get('last_result_payload', {}).get('refined_prompt', user_prompt)}\n\n"
            f"**Total Attempts:** {state.get('retry_count', 0) + 1}"
        )
        return (
            cur_image, state,
            f"✅ Score {int_score}/5 saved. Iteration complete!",
            gr.update(visible=False),
            gr.update(visible=True),
            details, face_st,
            load_history_modal_html()
        )

    retry_count   = state.get("retry_count", 0)
    booster       = PROMPT_BOOSTERS[min(retry_count, len(PROMPT_BOOSTERS) - 1)]
    base_prompt   = user_prompt
    if feedback and feedback.strip():
        base_prompt = f"{user_prompt}, {feedback.strip()}"
    boosted_prompt = base_prompt + booster
    state["retry_count"] = retry_count + 1

    orig_image = state.get("orig_image")
    if orig_image is None:
        orig_image = cur_image
    session_id = state.get("session_id", str(uuid.uuid4()))
    iteration  = int(state.get("iteration", 1))
    source_id    = state.get("source_id")
    dataset_name = state.get("dataset_name")
    file_name    = state.get("file_name")
    image_path   = state.get("image_path")

    mode_label = "🔄 Auto-Refinement"
    if is_enhancement_request(boosted_prompt):
        out, refined_prompt, neg_prompt, strength = generate_enhancement(
            cur_image, boosted_prompt, model_name, seed=42 + iteration
        )
        mask_region = "enhancement"
    else:
        out, refined_prompt, neg_prompt, mask_region, strength = generate_identity_preserving(
            cur_image, boosted_prompt, model_name, seed=42 + iteration
        )

    bench = benchmark_face_change(orig_image, out)
    vlm_score = get_vlm_score(out)
    uid   = str(uuid.uuid4())[:8]
    path  = os.path.join(SAVE_DIR, f"{uid}.png")
    out.save(path)
    now_ts = datetime.now().isoformat()

    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        INSERT INTO results (
            id, session_id, source_id, input_mode, iteration,
            base_prompt, refined_prompt, realism_score, vlm_score, scorer_id,
            model_name, image_path, timestamp,
            identity_sim, face_found, landmark_shift_mean, landmark_shift_max,
            landmark_count_before, landmark_count_after, identity_changed, benchmark_note
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        uid, session_id, source_id, input_mode, iteration,
        boosted_prompt, refined_prompt, None, vlm_score, scorer_id,
        model_name, path, now_ts,
        bench["identity_sim"], bench["face_found"],
        bench["landmark_shift_mean"], bench["landmark_shift_max"],
        bench["landmark_count_before"], bench["landmark_count_after"],
        bench["identity_changed"], bench["benchmark_note"]
    ))
    conn.commit(); conn.close()

    result_payload = {
        "id": uid, "session_id": session_id, "source_id": source_id,
        "input_mode": input_mode, "iteration": iteration,
        "base_prompt": boosted_prompt, "refined_prompt": refined_prompt,
        "model_name": model_name, "image_path": path, "timestamp": now_ts, "vlm_score": vlm_score, **bench
    }
    state.update({
        "iteration":           iteration + 1,
        "last_result_id":      uid,
        "last_result_payload": result_payload,
        "score_saved":         False,
        "source_id":           source_id,
        "dataset_name":        dataset_name,
        "file_name":           file_name,
        "image_path":          image_path,
        "input_mode":          input_mode,
        "orig_image":          orig_image,
    })

    if bench["face_found"] == 1:
        sim_v = bench["identity_sim"]
        face_status = f"Face Identity Tracker: **✅** — ArcFace: `{sim_v:.3f}`"
        face_status += " ✔️ Same" if bench["identity_changed"] == 0 else " ⚠️ Drifted"
    else:
        face_status = "Face Identity Tracker: **—**"
    state["last_face_status"] = face_status

    sim_display  = f"`{bench['identity_sim']:.3f}`" if bench['face_found'] == 1 else "`N/A`"
    details = f"""
**Mode:** {mode_label} (attempt {state['retry_count']})

**Model Engine:** {model_name}

**Previous Score:** {int_score}/5 — auto-refining...

**Boosted Prompt:** {refined_prompt}

**ArcFace Identity Similarity:** {sim_display}

**VLM Generated Score:** `{vlm_score}/5`
"""
    status_msg = (
        f"🔄 Score {int_score}/5 saved → Auto-refining "
        f"(attempt {state['retry_count']})... Rate again when done."
    )
    return (
        out, state, status_msg,
        gr.update(visible=True),
        gr.update(visible=True, value=3),
        details, face_status,
        load_history_modal_html()
    )

# ─────────────────────────────────────────────────
# BATCH MODE
# ─────────────────────────────────────────────────

ALL_PROMPTS_FLAT = [p for cat in PROMPT_CATEGORIES.values() for p in cat if p.strip()]

def load_n_random_faces(n: int):
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("""
        SELECT source_id, image_path, dataset_name, file_name
        FROM source_faces WHERE is_active=1
        ORDER BY RANDOM() LIMIT ?
    """, (int(n),)).fetchall()
    conn.close()
    return rows

def build_batch_progress_html(batch_state: dict) -> str:
    if not batch_state or "batch_ids" not in batch_state:
        return ""
    total   = len(batch_state["batch_ids"])
    scored  = batch_state.get("batch_scored_count", 0)
    cur_idx = batch_state.get("batch_cursor", 0)

    circles = []
    for i in range(total):
        if i < scored:
            col = "#00CC88"
        elif i == cur_idx:
            col = "#F4BB44"
        else:
            col = "#2a3550"
        circles.append(
            f'<span style="display:inline-block;width:10px;height:10px;'
            f'border-radius:50%;background:{col};margin:1px;"></span>'
        )

    rows_html = ""
    for chunk_start in range(0, total, 25):
        rows_html += "<div>" + "".join(circles[chunk_start:chunk_start+25]) + "</div>"

    return (
        f'<div style="background:#131929;border:1px solid #F4BB4444;border-radius:8px;'
        f'padding:10px;margin-top:8px;">'
        f'<div style="color:#F4BB44;font-weight:bold;margin-bottom:6px;">'
        f'Progress: {scored}/{total} scored'
        f'{"  ·  " + str(total - scored) + " remaining" if scored < total else "  ✅ All done!"}'
        f'</div>'
        f'{rows_html}</div>'
    )

def run_batch_generation(batch_prompt, batch_model, batch_size, scorer_id, batch_state):
    batch_state = dict(batch_state or {})
    n = int(batch_size)

    if not batch_prompt or not batch_prompt.strip():
        yield (
            batch_state,
            "⚠️ Please select a prompt first.",
            None, None,
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value="⚠️ No prompt selected.")
        )
        return

    face_rows = load_n_random_faces(n)
    if not face_rows:
        yield (
            batch_state,
            "❌ No source faces found. Run PREPARE FACE DB first.",
            None, None,
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value="❌ No faces in database.")
        )
        return

    actual_n    = len(face_rows)
    batch_ids   = []
    session_id  = str(uuid.uuid4())

    for idx, (source_id, image_path, dataset_name, file_name) in enumerate(face_rows):
        try:
            orig_img = Image.open(image_path).convert("RGB")
        except Exception as e:
            continue

        seed = 42 + idx
        try:
            if is_enhancement_request(batch_prompt):
                out, refined_prompt, neg_prompt, strength = generate_enhancement(
                    orig_img, batch_prompt, batch_model, seed=seed
                )
                mask_region = "enhancement"
            else:
                out, refined_prompt, neg_prompt, mask_region, strength = generate_identity_preserving(
                    orig_img, batch_prompt, batch_model, seed=seed
                )
        except Exception as e:
            refined_prompt = batch_prompt
            out = orig_img
            mask_region = "error"

        bench  = benchmark_face_change(orig_img, out)
        vlm_score = get_vlm_score(out)
        uid    = str(uuid.uuid4())[:8]
        out_path = os.path.join(SAVE_DIR, f"batch_{uid}.png")
        out.save(out_path)
        now_ts = datetime.now().isoformat()

        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT INTO results (
                id, session_id, source_id, input_mode, iteration,
                base_prompt, refined_prompt, realism_score, vlm_score, scorer_id,
                model_name, image_path, timestamp,
                identity_sim, face_found, landmark_shift_mean, landmark_shift_max,
                landmark_count_before, landmark_count_after, identity_changed, benchmark_note
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            uid, session_id, source_id, "Batch", 1,
            batch_prompt, refined_prompt, None, vlm_score, scorer_id,
            batch_model, out_path, now_ts,
            bench["identity_sim"], bench["face_found"],
            bench["landmark_shift_mean"], bench["landmark_shift_max"],
            bench["landmark_count_before"], bench["landmark_count_after"],
            bench["identity_changed"], bench["benchmark_note"]
        ))
        conn.commit(); conn.close()

        batch_ids.append({
            "id": uid,
            "orig_image_path": image_path,
            "output_path": out_path,
            "source_id": source_id,
            "dataset_name": dataset_name,
            "file_name": file_name,
            "model_name": batch_model,
            "refined_prompt": refined_prompt,
            "base_prompt": batch_prompt,
            "retry_count": 0,
            "vlm_score": vlm_score,
            **bench
        })

        yield (
            batch_state,
            f"⚙️ Generating {idx+1}/{actual_n}... | Prompt: **{batch_prompt}** | Model: **{batch_model}**",
            orig_img,
            out,
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value=f"Generating {idx+1} / {actual_n}...")
        )

    batch_state.update({
        "batch_ids":          batch_ids,
        "batch_cursor":       0,
        "batch_scored_count": 0,
        "batch_prompt":       batch_prompt,
        "batch_model":        batch_model,
        "batch_session":      session_id,
    })

    if not batch_ids:
        yield (
            batch_state,
            "❌ No images were generated. Check face database.",
            None, None,
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value="❌ Generation failed.")
        )
        return

    first_entry = batch_ids[0]
    orig_img = Image.open(first_entry["orig_image_path"]).convert("RGB")
    out_img  = Image.open(first_entry["output_path"]).convert("RGB")
    done_msg = (
        f"✅ **Batch complete!** {len(batch_ids)} images generated.\n\n"
        f"**Prompt:** {batch_prompt} | **Model:** {batch_model}\n\n"
        f"Now rate each image 0–5 and click **SAVE & NEXT**. (Scores < 5 will Auto-Refine)"
    )
    yield (
        batch_state,
        done_msg,
        orig_img,
        out_img,
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(value=build_batch_progress_html(batch_state))
    )


def save_batch_score_action(score_value, scorer_id, batch_state):
    if not batch_state or "batch_ids" not in batch_state:
        return (batch_state, None, None,
                "⚠️ No batch loaded.", "", gr.update(visible=False),
                gr.update(visible=False), gr.update(value=""))

    batch_state = dict(batch_state)
    ids     = batch_state["batch_ids"]
    cursor  = batch_state.get("batch_cursor", 0)
    total   = len(ids)

    if cursor >= total:
        return (batch_state, None, None,
                "✅ All images in this batch have been scored!",
                "", gr.update(visible=False), gr.update(visible=False),
                gr.update(value=build_batch_progress_html(batch_state)))

    entry         = ids[cursor]
    numeric_score = float(int(round(float(score_value))))
    
    # ── AUTO REFINE LOGIC FOR BATCH ──
    if numeric_score < 5:
        retry_count = entry.get("retry_count", 0)
        booster = PROMPT_BOOSTERS[min(retry_count, len(PROMPT_BOOSTERS) - 1)]
        boosted_prompt = entry["base_prompt"] + booster
        
        orig_img = Image.open(entry["orig_image_path"]).convert("RGB")
        cur_img = Image.open(entry["output_path"]).convert("RGB")
        
        # Save the bad score for history tracking
        conn = sqlite3.connect(DB_PATH)
        conn.execute("UPDATE results SET realism_score=?, scorer_id=? WHERE id=?", (numeric_score, scorer_id, entry["id"]))
        conn.commit(); conn.close()

        # Re-generate
        if is_enhancement_request(boosted_prompt):
            out, refined_prompt, neg_prompt, strength = generate_enhancement(
                cur_img, boosted_prompt, entry["model_name"], seed=42 + retry_count + cursor
            )
        else:
            out, refined_prompt, neg_prompt, mask_region, strength = generate_identity_preserving(
                cur_img, boosted_prompt, entry["model_name"], seed=42 + retry_count + cursor
            )
            
        bench = benchmark_face_change(orig_img, out)
        vlm_score = get_vlm_score(out)
        uid   = str(uuid.uuid4())[:8]
        out_path  = os.path.join(SAVE_DIR, f"batch_{uid}.png")
        out.save(out_path)
        now_ts = datetime.now().isoformat()
        
        # Insert new refined entry
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT INTO results (
                id, session_id, source_id, input_mode, iteration,
                base_prompt, refined_prompt, realism_score, vlm_score, scorer_id,
                model_name, image_path, timestamp,
                identity_sim, face_found, landmark_shift_mean, landmark_shift_max,
                landmark_count_before, landmark_count_after, identity_changed, benchmark_note
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            uid, batch_state.get("batch_session", ""), entry.get("source_id"), "Batch", retry_count + 2,
            boosted_prompt, refined_prompt, None, vlm_score, scorer_id,
            entry["model_name"], out_path, now_ts,
            bench["identity_sim"], bench["face_found"],
            bench["landmark_shift_mean"], bench["landmark_shift_max"],
            bench["landmark_count_before"], bench["landmark_count_after"],
            bench["identity_changed"], bench["benchmark_note"]
        ))
        conn.commit(); conn.close()
        
        # Update entry dictionary with new file and increment retry
        ids[cursor].update({
            "id": uid,
            "output_path": out_path,
            "retry_count": retry_count + 1,
            "refined_prompt": refined_prompt,
            "vlm_score": vlm_score,
            **bench
        })
        batch_state["batch_ids"] = ids
        
        info = (
            f"**Image {cursor+1} of {total}** |   Scored: {batch_state.get('batch_scored_count', 0)}   |   Remaining: {total - batch_state.get('batch_scored_count', 0)}\n\n"
            f"**File:** {entry.get('file_name','?')}   |   **Prompt:** {refined_prompt[:80]}"
        )
        
        return (
            batch_state,
            orig_img, out,
            f"🔄 Score {int(numeric_score)}/5 saved. Auto-refining (attempt {retry_count + 1}). Rate the new image.",
            info,
            gr.update(visible=True),
            gr.update(visible=True),
            gr.update(value=build_batch_progress_html(batch_state))
        )

    # ── IF SCORE == 5 (or user forces next logic) ──
    conn = sqlite3.connect(DB_PATH)
    conn.execute("UPDATE results SET realism_score=?, scorer_id=? WHERE id=?", (numeric_score, scorer_id, entry["id"]))
    conn.commit(); conn.close()

    source_meta = {
        "source_id":    entry.get("source_id"),
        "dataset_name": entry.get("dataset_name"),
        "file_name":    entry.get("file_name"),
        "image_path":   entry.get("orig_image_path"),
    }
    upsert_benchmark_current(source_meta, entry["model_name"], {
        "id":            entry["id"],
        "session_id":    batch_state.get("batch_session", ""),
        "source_id":     entry.get("source_id"),
        "input_mode":    "Batch",
        "iteration":     entry.get("retry_count", 0) + 1,
        "base_prompt":   batch_state.get("batch_prompt", ""),
        "refined_prompt": entry.get("refined_prompt", ""),
        "model_name":    entry["model_name"],
        "image_path":    entry["output_path"],
        "timestamp":     datetime.now().isoformat(),
        "vlm_score":     entry.get("vlm_score", 0.0),
        **{k: entry.get(k) for k in [
            "identity_sim","face_found","landmark_shift_mean","landmark_shift_max",
            "landmark_count_before","landmark_count_after","identity_changed","benchmark_note"
        ]}
    }, numeric_score, scorer_id)

    batch_state["batch_cursor"]       = cursor + 1
    batch_state["batch_scored_count"] = batch_state.get("batch_scored_count", 0) + 1
    new_cursor = batch_state["batch_cursor"]

    if new_cursor >= total:
        return (
            batch_state,
            None, None,
            f"🎉 **All {total} images scored!** Great work. Export the database to save results.",
            f"{total} / {total} — Complete!",
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value=build_batch_progress_html(batch_state))
        )

    next_entry = ids[new_cursor]
    orig_img = Image.open(next_entry["orig_image_path"]).convert("RGB")
    out_img  = Image.open(next_entry["output_path"]).convert("RGB")
    scored   = batch_state["batch_scored_count"]
    info     = (
        f"**Image {new_cursor+1} of {total}** |   "
        f"Scored: {scored}   |   Remaining: {total - scored}\n\n"
        f"**File:** {next_entry.get('file_name','?')}   |   "
        f"**Prompt:** {next_entry.get('refined_prompt','')[:80]}"
    )
    return (
        batch_state,
        orig_img, out_img,
        f"✅ Saved {score_value}/5 → Image {new_cursor+1} of {total}",
        info,
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(value=build_batch_progress_html(batch_state))
    )

def bulk_save_remaining_action(default_score, scorer_id, batch_state):
    if not batch_state or "batch_ids" not in batch_state:
        return (batch_state, "⚠️ No batch loaded.",
                gr.update(value=build_batch_progress_html(batch_state or {})))

    batch_state   = dict(batch_state)
    ids           = batch_state["batch_ids"]
    cursor        = batch_state.get("batch_cursor", 0)
    numeric_score = float(int(round(float(default_score))))
    count         = 0

    conn = sqlite3.connect(DB_PATH)
    for entry in ids[cursor:]:
        conn.execute("UPDATE results SET realism_score=?, scorer_id=? WHERE id=?", (numeric_score, scorer_id, entry["id"]))
        count += 1
    conn.commit(); conn.close()

    batch_state["batch_cursor"]       = len(ids)
    batch_state["batch_scored_count"] = len(ids)

    return (
        batch_state,
        f"✅ **Bulk saved!** {count} remaining images scored as **{int(numeric_score)}/5** by {scorer_id}.",
        gr.update(value=build_batch_progress_html(batch_state))
    )

def batch_jump_action(direction, batch_state):
    if not batch_state or "batch_ids" not in batch_state:
        return batch_state, None, None, "No batch loaded.", ""
    batch_state = dict(batch_state)
    ids    = batch_state["batch_ids"]
    cursor = batch_state.get("batch_cursor", 0)
    total  = len(ids)

    if direction == "prev":
        cursor = max(0, cursor - 1)
    else:
        cursor = min(total - 1, cursor + 1)

    batch_state["batch_cursor"] = cursor
    entry  = ids[cursor]
    orig_img = Image.open(entry["orig_image_path"]).convert("RGB")
    out_img  = Image.open(entry["output_path"]).convert("RGB")
    scored = batch_state.get("batch_scored_count", 0)
    info   = (
        f"**Image {cursor+1} of {total}** |   Scored: {scored}   |   Remaining: {total - scored}\n\n"
        f"**File:** {entry.get('file_name','?')}"
    )
    return batch_state, orig_img, out_img, info, f"{cursor+1} / {total}"

# ─────────────────────────────────────────────────
# UI action helpers (Editor Mode)
# ─────────────────────────────────────────────────
def on_mode_change(mode):
    if mode == "Upload":
        return gr.update(visible=False), "ℹ️ Upload mode selected."
    return gr.update(visible=True), "ℹ️ Database mode. Click PREPARE FACE DB, then LOAD RANDOM FACE."

def reset_for_uploaded_image(img):
    img = ensure_pil(img)
    if img is None:
        return None, "⚠️ Upload an image first.", gr.update(visible=False), gr.update(value=0.6)
    state = {
        "session_id": str(uuid.uuid4()), "iteration": 1,
        "orig_image": img.copy(), "source_id": None,
        "input_mode": "Upload", "dataset_name": None,
        "file_name": None, "image_path": None,
        "last_result_id": None, "last_result_payload": None, "score_saved": False
    }
    return state, "✅ Image set as session source.", gr.update(visible=False), gr.update(value=3)

def prepare_face_db_action():
    return prepare_source_face_db(limit=5000), load_history_modal_html()

def load_db_face_action():
    img, meta, msg = load_random_source_face()
    if img is None:
        return (None, None, "", "⚠️ No source faces. Click PREPARE FACE DB first.",
                gr.update(visible=False), gr.update(value=3))
    new_state = {
        "session_id": str(uuid.uuid4()), "iteration": 1,
        "orig_image": img.copy(), "source_id": meta["source_id"],
        "input_mode": "Database", "dataset_name": meta.get("dataset_name"),
        "file_name": meta.get("file_name"), "image_path": meta.get("image_path"),
        "last_result_id": None, "last_result_payload": None, "score_saved": False
    }
    return (img, new_state, "", "✅ Random database face loaded.",
            gr.update(visible=False), gr.update(value=3))

# ─────────────────────────────────────────────────
# CSS
# ─────────────────────────────────────────────────
studio_css = """
@import url('https://fonts.googleapis.com/css2?family=Press+Start+2P&display=swap');
html, body {
    margin: 0 !important;
    padding: 0 !important;
    background: #0d1117 !important;
    color: #e6edf3 !important;
}

.gradio-container,
.gradio-container > .main,
footer {
    background: #0d1117 !important;
    color: #e6edf3 !important;
}

.gradio-container {
    background-image:
        linear-gradient(rgba(0,0,0,0.80), rgba(0,0,0,0.90)),
        url('https://images.unsplash.com/photo-1605810230434-7631ac76ec81?q=80&w=2070&auto=format&fit=crop') !important;
    background-size: cover !important;
    background-position: center top !important;
    background-attachment: scroll !important;
    min-height: 100vh !important;
}

:root {
    --background-fill-primary:    #161b22 !important;
    --background-fill-secondary:  #0d1117 !important;
    --border-color-primary:       #30363d !important;
    --border-color-accent:        #F4BB44 !important;
    --color-accent:               #F4BB44 !important;
    --body-text-color:            #e6edf3 !important;
    --body-text-color-subdued:    #8b949e !important;
    --input-background-fill:      #21262d !important;
    --input-border-color:         #30363d !important;
    --block-background-fill:      #161b22 !important;
    --block-label-background-fill:#21262d !important;
    --panel-background-fill:      #0d1117 !important;
    --slider-color:               #F4BB44 !important;
    --button-primary-background-fill:    #F4BB44 !important;
    --button-primary-text-color:         #000000 !important;
    --button-secondary-background-fill:  #21262d !important;
    --button-secondary-text-color:       #e6edf3 !important;
    --button-secondary-border-color:     #30363d !important;
    --checkbox-background-color:         #21262d !important;
    --checkbox-background-color-selected:#F4BB44 !important;
    --radio-circle-color:                #F4BB44 !important;
}

.block, .form, .gap, .padded,
input, textarea, select,
.gr-box, .gr-form, .gr-panel {
    background-color: #161b22 !important;
    color: #e6edf3 !important;
    border-color: #30363d !important;
}

label, .label-wrap span {
    color: #c9d1d9 !important;
}

.prose, .prose p, .prose li, .md {
    color: #c9d1d9 !important;
}

.pixel-title {
    font-family: 'Press Start 2P', cursive !important;
    font-size: clamp(1.4rem, 3.5vw, 3rem);
    color: #F4BB44;
    text-shadow: 4px 4px #b8860b, 8px 8px #000;
    margin: 12px 0 20px 0;
    text-align: center;
}

.pixel-btn {
    background: #F4BB44 !important;
    color: #000 !important;
    font-family: 'Press Start 2P' !important;
    font-size: 0.6rem !important;
    padding: 11px 18px !important;
    border: 3px solid rgba(255,255,255,0.5) !important;
    box-shadow: 0 5px 0 #b8860b !important;
    cursor: pointer !important;
    border-radius: 5px !important;
    transition: transform 0.1s, box-shadow 0.1s !important;
    width: 100%;
}
.pixel-btn:active {
    transform: translateY(3px) !important;
    box-shadow: 0 2px 0 #b8860b !important;
}

.glass-panel {
    background: rgba(13, 17, 23, 0.95) !important;
    backdrop-filter: blur(14px) !important;
    border: 1px solid rgba(244,187,68,0.30) !important;
    border-radius: 12px !important;
    padding: 18px !important;
}

.tip-box {
    background: rgba(244,187,68,0.07);
    border: 1px solid rgba(244,187,68,0.22);
    border-radius: 8px;
    padding: 12px 16px;
    margin-bottom: 14px;
    color: #c9d1d9;
    font-size: 0.87rem;
    line-height: 1.65;
}

.history-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(172px, 1fr));
    gap: 14px;
    padding: 10px 0;
}
.history-card {
    position: relative;
    border-radius: 10px;
    overflow: hidden;
    background: #0d1117;
    border: 2px solid #21262d;
    cursor: pointer;
    transition: transform .18s, box-shadow .18s, border-color .18s;
}
.history-card:hover {
    transform: translateY(-4px);
    border-color: #F4BB44;
    box-shadow: 0 0 20px rgba(244,187,68,0.28);
}
.history-card img { width: 100%; height: 186px; object-fit: cover; display: block; }
.card-info { padding: 8px; background: #0d1117; }
.card-model { color: #c9d1d9; font-weight: 700; margin-bottom: 3px; font-size: 0.84rem; }
.card-score { color: #8b949e; font-size: 0.81rem; }
.best-badge {
    position: absolute; top: 0; left: 0; right: 0;
    background: #00e676; color: #000;
    text-align: center;
    font-weight: 700;
    padding: 4px; z-index: 2; font-size: 0.78rem;
}
.history-modal {
    display: none; position: fixed; inset: 0;
    z-index: 99999;
    background: rgba(0,0,0,0.88);
    align-items: center; justify-content: center; padding: 16px;
}
.history-modal-content {
    width: 96vw; height: 92vh; overflow: auto;
    background: #0d1117;
    border: 2px solid #F4BB44;
    border-radius: 16px; padding: 20px; position: relative;
}
.history-close-btn {
    position: absolute; top: 12px; right: 12px;
    width: 36px; height: 36px; border-radius: 50%;
    border: none; background: #F4BB44; color: #000;
    font-size: 18px; font-weight: 700; cursor: pointer;
}
.history-modal-layout {
    display: grid; grid-template-columns: 1.45fr 1fr; gap: 18px; margin-top: 18px; height: calc(92vh - 68px);
}
.history-modal-left {
    display: flex; align-items: center; justify-content: center; background: #000; border-radius: 10px; overflow: hidden;
}
.history-modal-image { width: 100%; height: 100%; object-fit: contain; display: block; }
.history-modal-right { color: #e6edf3; line-height: 1.55; overflow: auto; padding-right: 6px; }
.history-modal-right h3 { color: #F4BB44; margin-top: 0; }
.history-modal-right div { margin-bottom: 5px; }
.history-prompt-box {
    margin-top: 8px;
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(244,187,68,0.20);
    border-radius: 8px; padding: 12px;
    white-space: pre-wrap; word-break: break-word;
    font-size: 0.84rem; color: #c9d1d9;
}
@media (max-width: 900px) {
    .history-modal-layout { grid-template-columns: 1fr; height: auto; }
    .history-modal-content { height: auto; max-height: 92vh; }
    .history-modal-image { max-height: 50vh; }
}
"""

# ─────────────────────────────────────────────────
# GRADIO UI
# ─────────────────────────────────────────────────
with gr.Blocks(css=studio_css, theme=gr.themes.Base()) as demo:
    state       = gr.State(None)
    batch_state = gr.State(None)

    gr.HTML('<div class="pixel-title">🎨 VISION STUDIO</div>')
    
    with gr.Row():
        scorer_id_input = gr.Textbox(label="🧑‍💻 Scorer / Guest Name (For Faculty Analysis)", value="Fahima", info="Enter name to track who is scoring the deepfakes")

    with gr.Tabs():
        # ========== EDITOR MODE ==========
        with gr.TabItem("✏️  Editor Mode"):
            gr.HTML("""
            <div class="tip-box">
            <b>💡 How it works:</b>
            1. Pick a <b>category + prompt</b>, choose a <b>model</b>, click <b>GENERATE</b>.
            2. Rate the output <b>0–5</b> using the slider.
            3. Click <b>SUBMIT SCORE &amp; AUTO-REFINE</b>.<br>
            • Score <b>0–3</b> = score auto-saved, system <b>auto-refines with a stronger prompt</b> instantly.<br>
            • Score <b>4 or 5</b> = accepted, iteration complete, saved to history.
            </div>
            """)
            with gr.Row():
                with gr.Column(elem_classes="glass-panel", scale=1):
                    input_mode = gr.Radio(choices=["Upload", "Database"], value="Upload", label="INPUT MODE")
                    with gr.Column(visible=False) as db_controls:
                        prepare_db_btn = gr.Button("🗂️  PREPARE FACE DB", elem_classes="pixel-btn")
                        load_db_btn    = gr.Button("🎲  LOAD RANDOM FACE", elem_classes="pixel-btn")
                    db_info   = gr.Markdown("")
                    input_img = gr.Image(label="INPUT SOURCE", height=300, type="pil")

                    gr.Markdown("---")
                    gr.Markdown("### 🎯  EDIT CONTROLS")
                    with gr.Row():
                        prompt_category = gr.Radio(choices=list(PROMPT_CATEGORIES.keys()), value="expressions", label="CATEGORY", scale=1)
                        prompt_dropdown = gr.Radio(choices=PROMPT_CATEGORIES["expressions"], value=PROMPT_CATEGORIES["expressions"][0], label="QUICK PROMPT", scale=2)
                    model = gr.Radio(choices=list(MODEL_REGISTRY.keys()), value="Realistic-Vision-V5.1", label="MODEL")
                    gr.Markdown("<span style='color:#9ca3af;font-size:0.82rem'>💡 <b>Realistic-Vision-V5.1</b> → best for face edits, skin, age &nbsp;|&nbsp; <b>Juggernaut-XL-v9</b> → best for high-quality output</span>")
                    gen_btn = gr.Button("▶  GENERATE", elem_classes="pixel-btn")
                    gr.Markdown("---")
                    with gr.Group(visible=False) as score_group:
                        gr.Markdown("### ⭐  RATE THIS OUTPUT  (0 = fake, 5 = perfectly real)")
                        gr.Markdown("<span style='color:#9ca3af;font-size:0.82rem'>0 = obvious AI &nbsp;|&nbsp; 1-2 = artefacts visible &nbsp;|&nbsp; 3 = acceptable &nbsp;|&nbsp; 4-5 = photorealistic<br><b style='color:#F4BB44'>Score 4 or 5 = accepted &amp; done. Score 0-3 = auto-saved and auto-refined instantly.</b></span>")
                        realism = gr.Slider(minimum=0, maximum=5, value=3, step=1, label="REALISM SCORE (0-5)")
                        submit_refine_btn = gr.Button("✔  SUBMIT SCORE & AUTO-REFINE", elem_classes="pixel-btn")
                    with gr.Group(visible=False) as refine_group:
                        gr.Markdown("---")
                        feedback = gr.Textbox(label="REFINEMENT / CUSTOM EDIT", placeholder="Type any edit: ‘add smoky eye makeup’, ‘make them look younger’ ...", lines=2)
                        refine_btn = gr.Button("♻️  MANUAL REFINE (type hint above)", elem_classes="pixel-btn")
                    status = gr.Markdown("`SYSTEM READY`")
                with gr.Column(elem_classes="glass-panel", scale=1):
                    output_img  = gr.Image(label="OUTPUT", height=500, interactive=False)
                    face_status = gr.Markdown("Face Identity Tracker: **—**")
                    gr.Markdown("### 📝  EXECUTION LOG")
                    details = gr.Markdown("No runs executed yet.")
            with gr.Column(elem_classes="glass-panel"):
                gr.Markdown("### 📂  SESSION ARCHIVE")
                with gr.Row():
                    refresh_history_btn = gr.Button("🔄  REFRESH", variant="secondary")
                    export_btn          = gr.Button("📦  EXPORT DATABASE", variant="primary")
                download_file = gr.File(label="DOWNLOAD ZIP", visible=False)
                history_html  = gr.HTML(load_history_modal_html())

            # Editor mode events
            input_mode.change(on_mode_change, inputs=input_mode, outputs=[db_controls, status])
            prepare_db_btn.click(prepare_face_db_action, outputs=[db_info, history_html])
            load_db_btn.click(load_db_face_action, outputs=[input_img, state, db_info, status, score_group, realism])
            input_img.change(reset_for_uploaded_image, inputs=input_img, outputs=[state, status, score_group, realism])
            prompt_category.change(fn=lambda cat: gr.update(choices=PROMPT_CATEGORIES[cat], value=PROMPT_CATEGORIES[cat][0]), inputs=prompt_category, outputs=prompt_dropdown)
            
            _inputs  = [input_mode, input_img, prompt_dropdown, model, realism, feedback, scorer_id_input, state]
            _outputs = [output_img, state, status, refine_group, score_group, details, face_status, history_html]
            gen_btn.click(iterate_action, inputs=_inputs, outputs=_outputs)
            refine_btn.click(iterate_action, inputs=_inputs, outputs=_outputs)
            _submit_inputs = [input_mode, input_img, prompt_dropdown, model, realism, feedback, scorer_id_input, state]
            submit_refine_btn.click(submit_and_refine_action, inputs=_submit_inputs, outputs=_outputs)
            refresh_history_btn.click(load_history_modal_html, outputs=history_html)
            export_btn.click(export_data, outputs=download_file)

        # ========== BATCH MODE ==========
        with gr.TabItem("🔄  Batch Mode"):
            gr.HTML("""
            <div class="tip-box">
            <b>💡 Batch Mode — How it works:</b>
            1. Pick a <b>prompt</b> and <b>model</b>, set a <b>batch size</b> (up to 100).
            2. Click <b>▶ RUN BATCH GENERATION</b> — system loops through random faces and generates each one.
            3. Once done, <b>rate each image</b> using the slider → <b>SAVE &amp; NEXT</b>. (<b>Scores under 5 will trigger Auto-Refinement!</b>)
            4. Use <b>BULK SAVE REMAINING</b> to stamp all unscored images at once.
            </div>
            """)
            with gr.Row():
                with gr.Column(scale=2):
                    batch_prompt_dd = gr.Dropdown(choices=ALL_PROMPTS_FLAT, value=ALL_PROMPTS_FLAT[0] if ALL_PROMPTS_FLAT else None, label="📝  BATCH PROMPT", info="This exact prompt will be applied to every image in the batch")
                with gr.Column(scale=1):
                    batch_model_dd = gr.Dropdown(choices=list(MODEL_REGISTRY.keys()), value="Realistic-Vision-V5.1", label="🤖  MODEL")
                with gr.Column(scale=1):
                    batch_size_sl = gr.Slider(minimum=5, maximum=100, value=20, step=5, label="📦  BATCH SIZE", info="Number of images to generate")
            batch_run_btn = gr.Button("▶  RUN BATCH GENERATION", elem_classes="pixel-btn", variant="primary")
            batch_gen_status = gr.Markdown("`Ready — configure above and click RUN BATCH`")

            with gr.Row():
                with gr.Column(scale=1):
                    batch_orig_img = gr.Image(label="📷 ORIGINAL (source face)", height=350, interactive=False)
                with gr.Column(scale=1):
                    batch_out_img = gr.Image(label="🎨 OUTPUT (edited)", height=350, interactive=False)

            with gr.Row():
                with gr.Column(scale=1):
                    batch_img_info = gr.Markdown("Select a prompt and run the batch to begin.")
                with gr.Column(scale=1):
                    batch_counter  = gr.Markdown("")

            with gr.Group(visible=False) as batch_score_group:
                gr.Markdown("### ⭐  RATE THIS IMAGE")
                gr.Markdown("<span style='color:#9ca3af;font-size:0.82rem'>0 = obvious AI artefacts &nbsp;|&nbsp; 3 = acceptable &nbsp;|&nbsp; 5 = photorealistic<br><b style='color:#F4BB44'>Scoring less than 5 will auto-refine the image!</b></span>")
                batch_score_sl = gr.Slider(minimum=0, maximum=5, value=3, step=1, label="SCORE (0-5)")
                batch_save_next_btn = gr.Button("💾  SAVE SCORE & NEXT  →", elem_classes="pixel-btn", variant="primary")

            with gr.Group(visible=False) as batch_nav_group:
                gr.Markdown("---")
                with gr.Row():
                    batch_prev_btn = gr.Button("← PREV", variant="secondary", scale=1)
                    batch_next_btn = gr.Button("NEXT →", variant="secondary", scale=1)
                gr.Markdown("<span style='color:#9ca3af;font-size:0.82rem'>Prev/Next lets you browse without saving a score</span>")
                gr.Markdown("---")
                batch_bulk_score_sl = gr.Slider(minimum=0, maximum=5, value=3, step=1, label="DEFAULT SCORE for bulk save")
                batch_bulk_save_btn = gr.Button("📦  BULK SAVE ALL REMAINING (same score)", variant="secondary")

            batch_progress_html = gr.HTML("")

            # Batch mode 
            batch_run_btn.click(
                fn=run_batch_generation,
                inputs=[batch_prompt_dd, batch_model_dd, batch_size_sl, scorer_id_input, batch_state],
                outputs=[
                    batch_state, batch_gen_status,
                    batch_orig_img, batch_out_img,
                    batch_score_group, batch_nav_group, batch_progress_html
                ]
            )
            batch_save_next_btn.click(
                fn=save_batch_score_action,
                inputs=[batch_score_sl, scorer_id_input, batch_state],
                outputs=[
                    batch_state, batch_orig_img, batch_out_img,
                    batch_gen_status, batch_img_info,
                    batch_score_group, batch_nav_group, batch_progress_html
                ]
            )
            batch_prev_btn.click(
                fn=lambda s: batch_jump_action("prev", s),
                inputs=[batch_state],
                outputs=[batch_state, batch_orig_img, batch_out_img, batch_img_info, batch_counter]
            )
            batch_next_btn.click(
                fn=lambda s: batch_jump_action("next", s),
                inputs=[batch_state],
                outputs=[batch_state, batch_orig_img, batch_out_img, batch_img_info, batch_counter]
            )
            batch_bulk_save_btn.click(
                fn=bulk_save_remaining_action,
                inputs=[batch_bulk_score_sl, scorer_id_input, batch_state],
                outputs=[batch_state, batch_gen_status, batch_progress_html]
            )

demo.queue().launch(
    share=True,
    allowed_paths=[SAVE_DIR, ZIP_PATH, CSV_PATH, XLSX_PATH,
                   DB_PATH, "/kaggle/input/datasets/sathyavgc/celeba-dataset-small/content/dataset_sample/images", "/kaggle/working"]
)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c18d74af6335b62817.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
